## Free Throw Model — Cross‑Session Evaluation & Calibration

### Prerequisites: 
* Run feature engineering scripts (extract_phase_features.py and prepare_phase_dataset.py) to get features. 

### Machine Learning Pipeline:
- #### **Step 1** Split data between training/validation and testing
    - Right now I’m using 80% of the data for training/validation and 20% for testing.
- #### **Step 2** Hyperparemeter Tuning 
    - k fold cross validation and a grid search to find the hyperparameters for each model that optimize f1 score
- #### **Step 3** Calibration/Thresholding on best CV models
    - two **calibration** methods:
        - sigmoid
        - isotonic 
    - two **thresholding** methods: 
        - highest F1 score possible
        - F1 score measured at the threshold where precision is constrained to 70%
- #### **Step 4** Final Training and Testing
    - train best performing models on all training data
    - plot confusion matrices, reliability curves, and precision recall curves 
- #### **Step 5** Diagnostics of Final Models
    - Test the model on the 20% testing data and see how it does on data that it has not seen before
- #### **Step 6** Feature Pruning
    - From permutation Importance

# TODO 
* Better Analysis/Interpretation
* Better Model Performance: (PR Curves, ROC curves, calibration curves)
* Feature Importance: coefficients (LogReg), impurity (RF/GB), SHAP/Permutation importances
* Error analysis: confusion matrix, misclassified samples.
* Save everything: model weights, plots, tables, config
* Make yaml config files control all configuration in this script 

In [58]:
# =================================================
# Create unique experiment directory 
# =================================================

from datetime import datetime
import re
from pathlib import Path
import yaml

PROJECT_ROOT = Path().resolve().parent # go up one level

# Load feature config
feature_cfg_path = PROJECT_ROOT / "feature_config.yaml"
with open(feature_cfg_path, "r") as f:
    feature_config = yaml.safe_load(f)

FEATURE_VERSION = feature_config["feature_version"]
MODEL_TYPE = feature_config["model_type"]   # "summary_stats" or "time_series"

# --- makes label for the experiment ---
def start_run(label: str = "baseline") -> Path:
    ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    
    # put MODEL_TYPE under experiments
    root = PROJECT_ROOT / "experiments" / MODEL_TYPE
    root.mkdir(parents=True, exist_ok=True)
    
    existing = sorted(root.glob(f"{ts}_run-*_{label}"))
    idx = 1
    if existing:
        m = re.search(r"_run-(\d{3})_", existing[-1].name)
        idx = (int(m.group(1)) + 1) if m else 1
    
    rd = root / f"{ts}_run-{idx:03d}_{label}"
    rd.mkdir(parents=True, exist_ok=True)
    print(f"[INFO] Run directory: {rd}")
    return rd

run_dir = start_run("baseline")

[INFO] Run directory: /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-26_14-05-56_run-001_baseline


In [59]:
# =================================================
# Step 0: Import Libraries and Create Models
# =================================================

from __future__ import annotations
from pathlib import Path
from joblib import dump, load

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import json
import re

# --- sklearn ---
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score as _f1_for_perm
from sklearn.model_selection import GridSearchCV, StratifiedKFold, GroupKFold
from sklearn.metrics import (
    precision_recall_fscore_support, average_precision_score,
    roc_auc_score, brier_score_loss
)
from sklearn.metrics import (
    precision_score, recall_score, f1_score, 
    average_precision_score, roc_auc_score
)
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, brier_score_loss,
    confusion_matrix
)

# --- Plotting Util (saves CM, PR, calibration) ---
from plots import save_all_eval_figures

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score,
    brier_score_loss, confusion_matrix
)

# --- Load Project Config ---
project_cfg_path = PROJECT_ROOT / "project_config.yaml"
with open(project_cfg_path, "r") as f:
    project_config = yaml.safe_load(f)

ATHLETE = project_config["athlete"]
SESSION = project_config["session"]


# --- Load Feature Config ---
feature_cfg_path = PROJECT_ROOT / "feature_config.yaml"
with open(feature_cfg_path, "r") as f:
    feature_config = yaml.safe_load(f)

FEATURE_VERSION = feature_config["feature_version"] # version of feature data set 
MODEL_TYPE = feature_config["model_type"] # summary_stats or time_series

# --- Paths and Directories ---
DATA_ROOT = PROJECT_ROOT / "data" / ATHLETE

SESSIONS = {
    "session_01": DATA_ROOT / "session_01" / "analysis" / "datasets" / "features" / MODEL_TYPE / FEATURE_VERSION,
    "session_02": DATA_ROOT / "session_02" / "analysis" / "datasets" / "features" / MODEL_TYPE / FEATURE_VERSION,
    "session_03": DATA_ROOT / "session_03" / "analysis" / "datasets" / "features" / MODEL_TYPE / FEATURE_VERSION,
    "session_04": DATA_ROOT / "session_04" / "analysis" / "datasets" / "features" / MODEL_TYPE / FEATURE_VERSION
    # add more sessions here...
}

In [60]:
# =================================================
# Step 1: Split data into train/validation and test
# =================================================

TEST_SIZE = 0.2
RANDOM_STATE = 42

X_train_list, X_test_list = [], []
y_train_list, y_test_list = [], []
session_train_list, session_test_list = [], []

for session_name, base in SESSIONS.items():
    # Load
    X = pd.read_csv(base / "X.csv")
    y = pd.read_csv(base / "y.csv").squeeze("columns").astype(int)

    # Stratify class proportions based on y (makes/misses)
    strat = y if y.nunique() > 1 else None
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=strat 
    )

    # Store with session labels
    X_train_list.append(X_tr) 
    X_test_list.append(X_te)
    y_train_list.append(y_tr)
    y_test_list.append(y_te)
    session_train_list.extend([session_name] * len(y_tr))
    session_test_list.extend([session_name] * len(y_te))

# Concatenate everything
X_train = pd.concat(X_train_list, ignore_index=True)
X_test  = pd.concat(X_test_list, ignore_index=True)
y_train = pd.concat(y_train_list, ignore_index=True)
y_test  = pd.concat(y_test_list, ignore_index=True)
session_train = pd.Series(session_train_list, name="session")
session_test  = pd.Series(session_test_list, name="session")

print("[INFO] Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("\n[TRAIN counts]")
print(pd.crosstab(session_train, y_train))
print("\n[TEST counts]")
print(pd.crosstab(session_test, y_test))


[INFO] Train shape: (399, 3) Test shape: (101, 3)

[TRAIN counts]
_label       0    1
session            
session_01  10    5
session_02  27   52
session_03  53   92
session_04  50  110

[TEST counts]
_label       0   1
session           
session_01   3   1
session_02   7  13
session_03  14  23
session_04  13  27


In [61]:
# =================================================
# Step 2: Pick Models and tune hyperparameters
# =================================================

try:
    groups = session_train   
except NameError:
    groups = None

MODELS = {
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ),
    "LinearSVC": LinearSVC(
        C=1.0,
        max_iter=10000,
        tol=1e-3,
        class_weight="balanced"
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42
        # no class_weight here
    ),
    "GaussianNB": GaussianNB(),

    # --- New (bare) estimators; scale in the pipeline ---
    "SVC_rbf": SVC(
        C=1.0,
        kernel="rbf",
        gamma="scale",
        class_weight="balanced",
        probability=True,     
        random_state=42
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=15,
        weights="distance",
        p=2
    ),
}

# Wrap models that need scaling; leave tree/NB raw
pipelines = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MODELS["LogisticRegression"])
    ]),
    "LinearSVC": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MODELS["LinearSVC"])
    ]),
    "RandomForest": Pipeline([
        ("clf", MODELS["RandomForest"])
    ]),
    "GradientBoosting": Pipeline([
        ("clf", MODELS["GradientBoosting"])
    ]),
    "GaussianNB": Pipeline([
        ("clf", MODELS["GaussianNB"])  # no scaling for NB
    ]),
    "SVC_rbf": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MODELS["SVC_rbf"])
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MODELS["KNN"])
    ]),
}

# Hyperparameter grids for each model
param_grids = {
    "LogisticRegression": {
        "clf__C": [0.1, 0.3, 1.0, 3.0, 10.0],
        "clf__penalty": ["l2"],
        "clf__solver": ["lbfgs"],
    },
    "LinearSVC": {
        "clf__C": [0.1, 0.3, 1.0, 3.0, 10.0],
        "clf__tol": [1e-3, 1e-4],
    },
    "RandomForest": {
        "clf__n_estimators": [200, 300, 500],
        "clf__max_depth": [None, 8, 16],
        "clf__min_samples_split": [2, 5],
        "clf__max_features": ["sqrt", "log2", None],
    },
    "GradientBoosting": {
        "clf__n_estimators": [100, 200, 300],
        "clf__learning_rate": [0.03, 0.1, 0.3],
        "clf__max_depth": [2, 3, 4],
        "clf__subsample": [0.7, 1.0],
    },
    "GaussianNB": {
        "clf__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6],
    },
    "SVC_rbf": {
        "clf__C": [0.3, 1.0, 3.0, 10.0],
        "clf__gamma": ["scale", 0.1, 0.03, 0.01],  # compact log-ish sweep
    },
    "KNN": {
        "clf__n_neighbors": [5, 11, 21, 31],
        "clf__weights": ["uniform", "distance"],
        "clf__p": [1, 2],  # Manhattan vs Euclidean
    },
}

# Cross Validation splitter 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# TODO - add GroupKFold option later down the road 

scoring = {
    "f1": "f1",
    "ap": "average_precision",
    "roc": "roc_auc"
}

leaderboard_rows = []
best_models = {}

for name, pipe in pipelines.items():
    print(f"\n=== Tuning {name} ===")

    supports_proba = hasattr(pipe, "predict_proba")
    scoring = {"roc": "roc_auc", "ap": "average_precision"}
    refit_metric = "roc"  # fallback if no probabilities (e.g., LinearSVC)

    if supports_proba:
        scoring.update({"nll": "neg_log_loss", "brier": "neg_brier_score"})
        refit_metric = "nll"  # pick by log loss (higher = better because it's negated)

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grids[name],
        cv=cv,
        scoring=scoring,
        refit=refit_metric,
        n_jobs=-1,
        verbose=0,
        return_train_score=False
    )
    grid.fit(X_train, y_train)
    best_models[name] = grid.best_estimator_

    res = grid.cv_results_
    def pick(key):
        k = f"mean_test_{key}"
        return res[k][grid.best_index_] if k in res else np.nan

    leaderboard_rows.append({
        "model": name,
        "best_params": grid.best_params_,
        "cv_primary": pick(refit_metric),  # neg_log_loss if available, else roc_auc
        "cv_nll": pick("nll"),
        "cv_brier": pick("brier"),
        "cv_roc": pick("roc"),
        "cv_ap": pick("ap"),
    })

leaderboard = (
    pd.DataFrame(leaderboard_rows)
    .sort_values("cv_primary", ascending=False)  # higher is better (neg_log_loss is negated)
    .reset_index(drop=True)
)
print("\n[LEADERBOARD — sorted by primary metric (neg_log_loss if available, else roc_auc)]")
display(leaderboard)


# Persist best models for this run
from joblib import dump
save_dir = run_dir / "cv_models"
save_dir.mkdir(parents=True, exist_ok=True)
for name, model in best_models.items():
    dump(model, save_dir / f"best_{name}.joblib")
print(f"[INFO] Saved tuned models to: {save_dir}")



=== Tuning LogisticRegression ===

=== Tuning LinearSVC ===

=== Tuning RandomForest ===

=== Tuning GradientBoosting ===

=== Tuning GaussianNB ===

=== Tuning SVC_rbf ===

=== Tuning KNN ===

[LEADERBOARD — sorted by primary metric (neg_log_loss if available, else roc_auc)]


,model,best_params,cv_primary,cv_nll,cv_brier,cv_roc,cv_ap
0,LinearSVC,"{'clf__C': 0.1, 'clf__tol': 0.001}",0.417682,NaN,NaN,0.417682,0.626650
1,SVC_rbf,"{'clf__C': 3.0, 'clf__gamma': 0.01}",-0.642270,-0.642270,-0.225093,0.417257,0.617989
2,KNN,"{'clf__n_neighbors': 31, 'clf__p': 1, 'clf__we...",-0.669679,-0.669679,-0.237123,0.496915,0.650621
3,GradientBoosting,"{'clf__learning_rate': 0.03, 'clf__max_depth':...",-0.674947,-0.674947,-0.238868,0.468377,0.635758
4,LogisticRegression,"{'clf__C': 0.1, 'clf__penalty': 'l2', 'clf__so...",-0.702162,-0.702162,-0.254445,0.417407,0.626586
5,RandomForest,"{'clf__max_depth': 8, 'clf__max_features': 'sq...",-0.727723,-0.727723,-0.261617,0.491034,0.641325
6,GaussianNB,{'clf__var_smoothing': 1e-06},-0.773642,-0.773642,-0.260230,0.483188,0.669794


[INFO] Saved tuned models to: /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-26_14-05-56_run-001_baseline/cv_models


In [62]:
# =================================================
# Step 3: Calibration & Threshold Tuning (probability-first)
# =================================================
from sklearn.metrics import (
    log_loss, brier_score_loss, average_precision_score, roc_auc_score,
    precision_score, recall_score, f1_score
)
import numpy as np
import pandas as pd

# Split training data into training/validation
X_subtr, X_val, y_subtr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

def brier_skill_score(y_true, y_prob):
    p_base = np.mean(y_true)
    brier_model = brier_score_loss(y_true, y_prob)
    brier_base  = p_base * (1 - p_base) + 1e-12
    return 1.0 - (brier_model / brier_base)

def expected_calibration_error(y_true, y_prob, n_bins=15):
    bins = np.linspace(0, 1, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        idx = (y_prob >= lo) & (y_prob < hi if i < n_bins-1 else y_prob <= hi)
        if idx.any():
            ece += idx.mean() * abs(y_prob[idx].mean() - y_true[idx].mean())
    return float(ece)

def pick_threshold(p, y, precision_floor=None, tmin=0.05, tmax=0.95, n=181):
    # (kept for reference; not used to *select* calibrations anymore)
    ts = np.linspace(tmin, tmax, n)
    best = {"t": 0.5, "precision": 0.0, "recall": 0.0, "f1": -1.0}
    for t in ts:
        pred = (p >= t).astype(int)
        prec = precision_score(y, pred, zero_division=0)
        rec  = recall_score(y, pred, zero_division=0)
        f1   = f1_score(y, pred, zero_division=0)
        if precision_floor is None:
            if f1 > best["f1"]:
                best = {"t": float(t), "precision": prec, "recall": rec, "f1": f1}
        else:
            if prec >= precision_floor and f1 > best["f1"]:
                best = {"t": float(t), "precision": prec, "recall": rec, "f1": f1}
    if precision_floor is not None and best["f1"] < 0:
        return pick_threshold(p, y, precision_floor=None, tmin=tmin, tmax=tmax, n=n)
    return best

calib_methods = ["sigmoid", "isotonic"]
strategies = {
    "f1_max": {"precision_floor": None},
    "f1_at_p70": {"precision_floor": 0.70},
}

results = []

# Evaluate each model + calibration method
for model_name, base_model in best_models.items():
    for method in calib_methods:
        print(f"\n=== {model_name} | calibration={method} ===")

        # Calibrated wrapper -> always gives predict_proba
        cal = CalibratedClassifierCV(base_model, cv=5, method=method)
        cal.fit(X_subtr, y_subtr)

        p_val = cal.predict_proba(X_val)[:, 1]

        # Threshold strategies (kept for reporting)
        s1 = pick_threshold(p_val, y_val, **strategies["f1_max"])
        s2 = pick_threshold(p_val, y_val, **strategies["f1_at_p70"])

        # ---- Probability-first metrics (NEW) ----
        nll   = log_loss(y_val, p_val)                    # ↓ better
        brier = brier_score_loss(y_val, p_val)            # ↓ better
        bss   = brier_skill_score(y_val, p_val)           # ↑ better
        ece   = expected_calibration_error(y_val, p_val)  # ↓ better
        ap    = average_precision_score(y_val, p_val)
        roc   = roc_auc_score(y_val, p_val)

        # Record (two rows for the threshold strategies, sharing prob metrics)
        results.extend([
            {
                "model": model_name, "calibration": method, "strategy": "f1_max",
                "best_threshold": s1["t"], "precision": s1["precision"],
                "recall": s1["recall"], "f1": s1["f1"],
                "LogLoss": nll, "Brier": brier, "BSS": bss, "ECE": ece,
                "AP": ap, "ROC_AUC": roc
            },
            {
                "model": model_name, "calibration": method, "strategy": "f1_at_p70",
                "best_threshold": s2["t"], "precision": s2["precision"],
                "recall": s2["recall"], "f1": s2["f1"],
                "LogLoss": nll, "Brier": brier, "BSS": bss, "ECE": ece,
                "AP": ap, "ROC_AUC": roc
            }
        ])

# Build comparison table
calib_threshold_df = (
    pd.DataFrame(results)
    .sort_values(by=["model", "calibration", "strategy"])
    .reset_index(drop=True)
)

print("\n[CALIBRATION / THRESHOLD COMPARISON — probability metrics included]")
display(calib_threshold_df)

# ---- Probability-first summary (wide): show thresholds & P/R/F1 per strategy ----
summary_rows = []
for m in calib_threshold_df["model"].unique():
    sub = calib_threshold_df[calib_threshold_df["model"] == m]

    # pick the best calibration by probability quality (ties -> Brier -> ECE)
    cal_rank = (
        sub.groupby("calibration", as_index=False)
           .agg(LogLoss=("LogLoss","mean"),
                Brier=("Brier","mean"),
                ECE=("ECE","mean"),
                BSS=("BSS","mean"),
                ROC_AUC=("ROC_AUC","mean"),
                AP=("AP","mean"))
           .sort_values(["LogLoss","Brier","ECE"], ascending=[True, True, True])
    )
    best_cal = cal_rank.iloc[0]["calibration"]

    # rows for the chosen calibration, indexed by strategy
    csub = sub[sub["calibration"] == best_cal].set_index("strategy")

    # pull shared prob metrics (same across strategies for a given cal)
    base = csub.iloc[0]

    row = {
        "model": m,
        "calibration": best_cal,
        "LogLoss": float(base["LogLoss"]),
        "Brier": float(base["Brier"]),
        "ECE": float(base["ECE"]),
        "BSS": float(base["BSS"]),
        "ROC_AUC": float(base["ROC_AUC"]),
        "AP": float(base["AP"]),
        # --- Strategy: f1_max ---
        "thr_f1_max": float(csub.loc["f1_max"]["best_threshold"]),
        "prec_f1_max": float(csub.loc["f1_max"]["precision"]),
        "rec_f1_max": float(csub.loc["f1_max"]["recall"]),
        "f1_f1_max": float(csub.loc["f1_max"]["f1"]),
        # --- Strategy: f1_at_p70 ---
        "thr_p70": float(csub.loc["f1_at_p70"]["best_threshold"]),
        "prec_p70": float(csub.loc["f1_at_p70"]["precision"]),
        "rec_p70": float(csub.loc["f1_at_p70"]["recall"]),
        "f1_p70": float(csub.loc["f1_at_p70"]["f1"]),
    }
    summary_rows.append(row)

summary = (
    pd.DataFrame(summary_rows)
      .sort_values(["LogLoss","Brier","ECE"], ascending=[True, True, True])
      .reset_index(drop=True)
)

print("\n[TOP CALIBRATION PER MODEL — full details (probability metrics + thresholds & P/R/F1)]")
display(summary)

# (Optional) save the wider summary too
summary.to_csv(run_dir / "calibration_threshold_summary_wide.csv", index=False)


# Save to run dir
calib_threshold_df.to_csv(run_dir / "calibration_threshold_grid.csv", index=False)
summary.to_csv(run_dir / "calibration_threshold_summary.csv", index=False)
print(f"[INFO] Saved: {run_dir/'calibration_threshold_grid.csv'} and summary")



=== LogisticRegression | calibration=sigmoid ===

=== LogisticRegression | calibration=isotonic ===

=== LinearSVC | calibration=sigmoid ===

=== LinearSVC | calibration=isotonic ===

=== RandomForest | calibration=sigmoid ===

=== RandomForest | calibration=isotonic ===

=== GradientBoosting | calibration=sigmoid ===

=== GradientBoosting | calibration=isotonic ===

=== GaussianNB | calibration=sigmoid ===

=== GaussianNB | calibration=isotonic ===

=== SVC_rbf | calibration=sigmoid ===

=== SVC_rbf | calibration=isotonic ===

=== KNN | calibration=sigmoid ===

=== KNN | calibration=isotonic ===

[CALIBRATION / THRESHOLD COMPARISON — probability metrics included]


,model,calibration,strategy,best_threshold,precision,recall,f1,LogLoss,Brier,BSS,ECE,AP,ROC_AUC
0,GaussianNB,isotonic,f1_at_p70,0.715,0.777778,0.134615,0.229508,0.686630,0.246835,-0.084988,0.193286,0.668466,0.450206
1,GaussianNB,isotonic,f1_max,0.050,0.650000,1.000000,0.787879,0.686630,0.246835,-0.084988,0.193286,0.668466,0.450206
2,GaussianNB,sigmoid,f1_at_p70,0.050,0.650000,1.000000,0.787879,0.650993,0.229062,-0.006867,0.036476,0.597282,0.413462
3,GaussianNB,sigmoid,f1_max,0.050,0.650000,1.000000,0.787879,0.650993,0.229062,-0.006867,0.036476,0.597282,0.413462
4,GradientBoosting,isotonic,f1_at_p70,0.050,0.650000,1.000000,0.787879,0.712087,0.257073,-0.129993,0.205791,0.566935,0.350962
5,GradientBoosting,isotonic,f1_max,0.050,0.650000,1.000000,0.787879,0.712087,0.257073,-0.129993,0.205791,0.566935,0.350962
6,GradientBoosting,sigmoid,f1_at_p70,0.680,0.750000,0.057692,0.107143,0.656602,0.231697,-0.018448,0.022256,0.614722,0.401099
7,GradientBoosting,sigmoid,f1_max,0.050,0.650000,1.000000,0.787879,0.656602,0.231697,-0.018448,0.022256,0.614722,0.401099
8,KNN,isotonic,f1_at_p70,0.740,0.727273,0.153846,0.253968,0.698803,0.250526,-0.101215,0.200330,0.618187,0.393544
9,KNN,isotonic,f1_max,0.050,0.650000,1.000000,0.787879,0.698803,0.250526,-0.101215,0.200330,0.618187,0.393544



[TOP CALIBRATION PER MODEL — full details (probability metrics + thresholds & P/R/F1)]


,model,calibration,LogLoss,Brier,ECE,BSS,ROC_AUC,AP,thr_f1_max,prec_f1_max,rec_f1_max,f1_f1_max,thr_p70,prec_p70,rec_p70,f1_p70
0,SVC_rbf,isotonic,0.644845,0.226828,0.074911,0.002954,0.491071,0.679122,0.51,0.658228,1.0,0.793893,0.71,0.777778,0.134615,0.229508
1,LinearSVC,sigmoid,0.645662,0.226652,0.022649,0.003727,0.496566,0.616717,0.60,0.658228,1.0,0.793893,0.60,0.658228,1.000000,0.793893
2,LogisticRegression,sigmoid,0.645782,0.226709,0.022716,0.003476,0.496566,0.616328,0.60,0.658228,1.0,0.793893,0.60,0.658228,1.000000,0.793893
3,RandomForest,sigmoid,0.646852,0.227235,0.006644,0.001164,0.534341,0.695273,0.05,0.650000,1.0,0.787879,0.65,0.736842,0.269231,0.394366
4,GaussianNB,sigmoid,0.650993,0.229062,0.036476,-0.006867,0.413462,0.597282,0.05,0.650000,1.0,0.787879,0.05,0.650000,1.000000,0.787879
5,GradientBoosting,sigmoid,0.656602,0.231697,0.022256,-0.018448,0.401099,0.614722,0.05,0.650000,1.0,0.787879,0.68,0.750000,0.057692,0.107143
6,KNN,sigmoid,0.668409,0.237125,0.099157,-0.042308,0.408654,0.631833,0.05,0.650000,1.0,0.787879,0.71,0.857143,0.115385,0.203390


[INFO] Saved: /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-26_14-05-56_run-001_baseline/calibration_threshold_grid.csv and summary


In [63]:
# =================================================
# Step 4: Full Training & Testing (Evaluation)
# =================================================
from sklearn.metrics import (
    precision_score, recall_score, f1_score, average_precision_score,
    roc_auc_score, brier_score_loss, log_loss, confusion_matrix
)

# Use the train already built; keep test untouched
X_train_full = X_train
y_train_full = y_train

final_dir = run_dir / "final_eval"
final_dir.mkdir(parents=True, exist_ok=True)
print(f"[INFO] Saving final model evaluations to {final_dir}")

# Models to evaluate  (fix SVC name)
chosen_models = ["LogisticRegression", "RandomForest", "GaussianNB",
                 "LinearSVC", "GradientBoosting", "KNN", "SVC_rbf"]

# sanity checks
if 'best_models' not in globals():
    raise RuntimeError("best_models not found (run the hyperparameter tuning section first).")
if 'calib_threshold_df' not in globals():
    raise RuntimeError("calib_threshold_df not found (run the calibration/threshold section first).")

# ----- Probability-first selection settings -----
P_FLOOR = 0.70
R_FLOOR = 0.70

# Build: best calibration + threshold per model (probability-first)
cfg_by_model = {}
for m in chosen_models:
    sub = calib_threshold_df[calib_threshold_df["model"] == m]
    if sub.empty:
        print(f"[WARN] No calibration rows for {m}; skipping.")
        continue

    # Pick best calibration by probability quality: LogLoss -> Brier -> ECE (all ascending)
    cal_rank = (
        sub.groupby("calibration", as_index=False)
           .agg(LogLoss=("LogLoss","mean"),
                Brier=("Brier","mean"),
                ECE=("ECE","mean"),
                AP=("AP","mean"),
                ROC_AUC=("ROC_AUC","mean"))
           .sort_values(["LogLoss","Brier","ECE"], ascending=[True, True, True])
    )
    best_cal = cal_rank.iloc[0]["calibration"]

    # Within chosen calibration, pick threshold row
    csub = sub[sub["calibration"] == best_cal].copy()

    # 1) Prefer f1_at_p70 if it truly meets both floors
    pick = None
    if "f1_at_p70" in set(csub["strategy"]):
        r = csub[csub["strategy"] == "f1_at_p70"].iloc[0]
        if (r["precision"] >= P_FLOOR) and (r["recall"] >= R_FLOOR):
            pick = r

    # 2) Else maximize the weaker of (precision, recall); tie-break by F1, then AP, then ROC
    if pick is None:
        csub["minPR"] = csub[["precision","recall"]].min(axis=1)
        pick = csub.sort_values(["minPR","f1","AP","ROC_AUC"],
                                ascending=[False, False, False, False]).iloc[0]

    cfg_by_model[m] = {
        "calibration": pick["calibration"],
        "threshold": float(pick["best_threshold"]),
        "strategy": pick["strategy"],
        # record validation metrics (probability-first)
        "val_LogLoss": float(sub[sub["calibration"]==best_cal]["LogLoss"].mean()),
        "val_Brier": float(sub[sub["calibration"]==best_cal]["Brier"].mean()),
        "val_ECE": float(sub[sub["calibration"]==best_cal]["ECE"].mean()),
        "val_AP": float(sub[sub["calibration"]==best_cal]["AP"].mean()),
        "val_ROC_AUC": float(sub[sub["calibration"]==best_cal]["ROC_AUC"].mean()),
        # and the chosen operating-point stats
        "val_precision": float(pick["precision"]),
        "val_recall": float(pick["recall"]),
        "val_F1": float(pick["f1"]),
    }

final_results = []
saved_paths = []

for m in chosen_models:
    if m not in cfg_by_model:
        continue
    if m not in best_models:
        print(f"[WARN] No tuned pipeline in best_models for {m}; skipping.")
        continue

    base = best_models[m]
    method = cfg_by_model[m]["calibration"]
    thr    = cfg_by_model[m]["threshold"]

    print(f"\n=== {m}: train full (calibration={method}, thr={thr:.3f}) ===")
    cal = CalibratedClassifierCV(base, cv=5, method=method)
    cal.fit(X_train_full, y_train_full)

    # predict on test
    p_test = cal.predict_proba(X_test)[:, 1]
    y_hat  = (p_test >= thr).astype(int)

    # metrics (probability-first + operating-point)
    nll  = log_loss(y_test, p_test)                 # ↓ better
    bri  = brier_score_loss(y_test, p_test)         # ↓ better
    ap   = average_precision_score(y_test, p_test)
    roc  = roc_auc_score(y_test, p_test)

    prec = precision_score(y_test, y_hat, zero_division=0)
    rec  = recall_score(y_test, y_hat, zero_division=0)
    f1   = f1_score(y_test, y_hat, zero_division=0)
    cm   = confusion_matrix(y_test, y_hat)

    print(f"[TEST] {m}  P={prec:.3f} R={rec:.3f} F1={f1:.3f} "
          f"AP={ap:.3f} ROC={roc:.3f} LogLoss={nll:.3f} Brier={bri:.3f}")
    print(pd.DataFrame(cm, index=["true 0","true 1"], columns=["pred 0","pred 1"]))

    # save model + plots
    model_path = final_dir / f"{m}_cal-{method}.joblib"
    dump(cal, model_path)
    saved_paths.append(model_path.as_posix())

    fig_dir = final_dir / f"{m}_figs"
    save_all_eval_figures(
        outdir=fig_dir,
        y_true=y_test,
        y_pred=y_hat,
        y_proba=p_test,
        class_names=["neg","pos"]  # or ["miss","make"]
    )

    final_results.append({
        "model": m,
        "calibration": method,
        "threshold": thr,
        "val_metrics": {
            "LogLoss": cfg_by_model[m]["val_LogLoss"],
            "Brier": cfg_by_model[m]["val_Brier"],
            "ECE": cfg_by_model[m]["val_ECE"],
            "AP": cfg_by_model[m]["val_AP"],
            "ROC_AUC": cfg_by_model[m]["val_ROC_AUC"],
            "precision": cfg_by_model[m]["val_precision"],
            "recall": cfg_by_model[m]["val_recall"],
            "F1": cfg_by_model[m]["val_F1"],
            "strategy": cfg_by_model[m]["strategy"],
        },
        "test_metrics": {
            "LogLoss": float(nll),
            "Brier": float(bri),
            "AP": float(ap),
            "ROC_AUC": float(roc),
            "precision": float(prec),
            "recall": float(rec),
            "F1": float(f1),
            "confusion_matrix": cm.tolist(),
        }
    })

# save summary
if final_results:
    final_df = pd.DataFrame([{
        "model": r["model"],
        "calibration": r["calibration"],
        "threshold": r["threshold"],
        "val_LogLoss": r["val_metrics"]["LogLoss"],
        "val_Brier": r["val_metrics"]["Brier"],
        "val_ECE": r["val_metrics"]["ECE"],
        "val_precision": r["val_metrics"]["precision"],
        "val_recall": r["val_metrics"]["recall"],
        "val_F1": r["val_metrics"]["F1"],
        "test_LogLoss": r["test_metrics"]["LogLoss"],
        "test_Brier": r["test_metrics"]["Brier"],
        "test_precision": r["test_metrics"]["precision"],
        "test_recall": r["test_metrics"]["recall"],
        "test_F1": r["test_metrics"]["F1"],
        "test_AP": r["test_metrics"]["AP"],
        "test_ROC_AUC": r["test_metrics"]["ROC_AUC"],
    } for r in final_results]).sort_values("test_LogLoss", ascending=True).reset_index(drop=True)

    display(final_df)
    (final_dir / "final_results.json").write_text(json.dumps(final_results, indent=2))
    final_df.to_csv(final_dir / "final_results.csv", index=False)
    print(f"\n[INFO] Saved models:\n  - " + "\n  - ".join(saved_paths))
    print(f"[INFO] Results:\n  - {final_dir/'final_results.json'}\n  - {final_dir/'final_results.csv'}")
else:
    print("[WARN] Nothing trained—check chosen_models & that Step 3 produced calib_threshold_df rows.)")


[INFO] Saving final model evaluations to /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-26_14-05-56_run-001_baseline/final_eval

=== LogisticRegression: train full (calibration=sigmoid, thr=0.600) ===
[TEST] LogisticRegression  P=0.626 R=0.969 F1=0.761 AP=0.564 ROC=0.409 LogLoss=0.667 Brier=0.237
        pred 0  pred 1
true 0       0      37
true 1       2      62


/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")



=== RandomForest: train full (calibration=sigmoid, thr=0.050) ===
[TEST] RandomForest  P=0.634 R=1.000 F1=0.776 AP=0.639 ROC=0.413 LogLoss=0.676 Brier=0.241
        pred 0  pred 1
true 0       0      37
true 1       0      64


/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")



=== GaussianNB: train full (calibration=sigmoid, thr=0.050) ===
[TEST] GaussianNB  P=0.634 R=1.000 F1=0.776 AP=0.563 ROC=0.378 LogLoss=0.666 Brier=0.236
        pred 0  pred 1
true 0       0      37
true 1       0      64


...

/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")



=== SVC_rbf: train full (calibration=isotonic, thr=0.510) ===
[TEST] SVC_rbf  P=0.634 R=1.000 F1=0.776 AP=0.616 ROC=0.433 LogLoss=0.665 Brier=0.236
        pred 0  pred 1
true 0       0      37
true 1       0      64


/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")


,model,calibration,threshold,val_LogLoss,val_Brier,val_ECE,val_precision,val_recall,val_F1,test_LogLoss,test_Brier,test_precision,test_recall,test_F1,test_AP,test_ROC_AUC
0,KNN,sigmoid,0.05,0.668409,0.237125,0.099157,0.650000,1.0,0.787879,0.659649,0.233337,0.633663,1.00000,0.775758,0.634029,0.495777
1,SVC_rbf,isotonic,0.51,0.644845,0.226828,0.074911,0.658228,1.0,0.793893,0.664887,0.235631,0.633663,1.00000,0.775758,0.616481,0.433066
2,GaussianNB,sigmoid,0.05,0.650993,0.229062,0.036476,0.650000,1.0,0.787879,0.666301,0.236254,0.633663,1.00000,0.775758,0.563417,0.377534
3,LinearSVC,sigmoid,0.60,0.645662,0.226652,0.022649,0.658228,1.0,0.793893,0.666648,0.236506,0.626263,0.96875,0.760736,0.563559,0.408361
4,LogisticRegression,sigmoid,0.60,0.645782,0.226709,0.022716,0.658228,1.0,0.793893,0.666746,0.236552,0.626263,0.96875,0.760736,0.563782,0.408784
5,GradientBoosting,sigmoid,0.05,0.656602,0.231697,0.022256,0.650000,1.0,0.787879,0.673067,0.239087,0.633663,1.00000,0.775758,0.592268,0.439189
6,RandomForest,sigmoid,0.05,0.646852,0.227235,0.006644,0.650000,1.0,0.787879,0.676024,0.240954,0.633663,1.00000,0.775758,0.638845,0.412584



[INFO] Saved models:
  - /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-26_14-05-56_run-001_baseline/final_eval/LogisticRegression_cal-sigmoid.joblib
  - /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-26_14-05-56_run-001_baseline/final_eval/RandomForest_cal-sigmoid.joblib
  - /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-26_14-05-56_run-001_baseline/final_eval/GaussianNB_cal-sigmoid.joblib
  - /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-26_14-05-56_run-001_baseline/final_eval/LinearSVC_cal-sigmoid.joblib
  - /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-26_14-05-56_run-001_baseline/final_eval/GradientBoosting_cal-sigmoid.joblib
  - /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-26_14-05-56_run-001_baseline/final_eval/KNN_cal-sigmoid.joblib
  - /Users

In [64]:
# ============================================================
# STEP 5: Diagnostics of final models  (probability-first)
# ============================================================
from sklearn.metrics import (
    precision_score, recall_score, f1_score, average_precision_score,
    roc_auc_score, brier_score_loss, log_loss, confusion_matrix
)
from sklearn.inspection import permutation_importance

# --------------------------
# Paths & inputs
# --------------------------
final_dir = run_dir / "final_eval"
results_json = final_dir / "final_results.json"
if not results_json.exists():
    raise FileNotFoundError(f"Expected {results_json} from Step 4 at: {results_json}")

with results_json.open() as f:
    final_results = json.load(f)

X_train_full = X_train
y_train_full = y_train

diag_root = run_dir / "diagnostics_final"
diag_root.mkdir(parents=True, exist_ok=True)

# Which split to use for permutation importance
if 'X_val' in globals() and 'y_val' in globals():
    use_X_perm, use_y_perm, perm_where = X_val, y_val, "val"
else:
    use_X_perm, use_y_perm, perm_where = X_test, y_test, "test"
print(f"[INFO] Permutation importance will run on: {perm_where} set")

# --------------------------
# Helpers
# --------------------------
def _clf_and_scaler_from_pipeline(pipeline):
    """Return (clf, scaler_or_None). Assumes steps named 'scaler' and 'clf'."""
    if hasattr(pipeline, "named_steps"):
        scaler = pipeline.named_steps.get("scaler", None)
        clf    = pipeline.named_steps.get("clf", None)
        clf    = clf if clf is not None else pipeline
    else:
        scaler, clf = None, pipeline
    return clf, scaler

def _linear_coef_df(pipeline, feature_names):
    """Coefficients in ORIGINAL units if StandardScaler present."""
    clf, scaler = _clf_and_scaler_from_pipeline(pipeline)
    if not hasattr(clf, "coef_"):
        return None
    coef = np.ravel(clf.coef_)
    if scaler is not None and hasattr(scaler, "scale_"):
        scale = np.where(np.asarray(scaler.scale_) == 0, 1.0, np.asarray(scaler.scale_))
        coef = coef / scale
    df = pd.DataFrame({"feature": feature_names, "coef": coef})
    df["abs_coef"] = df["coef"].abs()
    return df.sort_values("abs_coef", ascending=False).reset_index(drop=True)

def _tree_importance_df(pipeline, feature_names):
    clf, _ = _clf_and_scaler_from_pipeline(pipeline)
    if not hasattr(clf, "feature_importances_"):
        return None
    imp = np.asarray(clf.feature_importances_, dtype=float)
    df = pd.DataFrame({"feature": feature_names, "impurity_importance": imp})
    return df.sort_values("impurity_importance", ascending=False).reset_index(drop=True)

def expected_calibration_error(y_true, y_prob, n_bins=15):
    bins = np.linspace(0, 1, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        idx = (y_prob >= lo) & (y_prob < hi if i < n_bins-1 else y_prob <= hi)
        if idx.any():
            ece += idx.mean() * abs(y_prob[idx].mean() - y_true[idx].mean())
    return float(ece)

def _permutation_importance_df(
    estimator, X_perm, y_perm, feature_names,
    mode="roc_auc",   # "roc_auc" | "average_precision" | "f1_at_threshold"
    threshold=None,
    n_repeats=30
):
    if mode in ("roc_auc", "average_precision"):
        scoring = mode
    elif mode == "f1_at_threshold":
        if threshold is None:
            raise ValueError("threshold must be provided for f1_at_threshold")
        def _score(est, X, y):
            p = est.predict_proba(X)[:, 1]
            yhat = (p >= threshold).astype(int)
            # F1 is fine here as a *diagnostic* at your operating point
            return f1_score(y, yhat, zero_division=0)
        scoring = _score
    else:
        raise ValueError(f"Unknown mode: {mode}")

    res = permutation_importance(
        estimator, X_perm, y_perm,
        scoring=scoring, n_repeats=n_repeats, random_state=42, n_jobs=-1
    )
    df = pd.DataFrame({
        "feature": feature_names,
        "perm_importance_mean": res.importances_mean,
        "perm_importance_std":  res.importances_std
    })
    return df.sort_values("perm_importance_mean", ascending=False).reset_index(drop=True)

# --------------------------
# Loop over final models
# --------------------------
summary_rows = []
feat_names = list(X_train_full.columns)

for entry in final_results:
    m        = entry["model"]
    method   = entry["calibration"]
    thr      = float(entry["threshold"])
    model_path = final_dir / f"{m}_cal-{method}.joblib"

    if not model_path.exists():
        print(f"[WARN] Missing saved model for {m} at {model_path}; skipping.")
        continue

    print(f"\n=== {m} | load final model | calibration={method} | threshold={thr:.3f} ===")
    cal = load(model_path)  # CalibratedClassifierCV

    # --- Performance on TEST (no refit) ---
    p_test = cal.predict_proba(X_test)[:, 1]
    y_hat  = (p_test >= thr).astype(int)

    nll  = log_loss(y_test, p_test)              # ↓ better
    bri  = brier_score_loss(y_test, p_test)      # ↓ better
    ap   = average_precision_score(y_test, p_test)
    roc  = roc_auc_score(y_test, p_test)
    ece  = expected_calibration_error(y_test, p_test)

    prec = precision_score(y_test, y_hat, zero_division=0)
    rec  = recall_score(y_test, y_hat, zero_division=0)
    f1   = f1_score(y_test, y_hat, zero_division=0)
    cm   = confusion_matrix(y_test, y_hat)

    print(f"[TEST] P={prec:.3f} R={rec:.3f} F1={f1:.3f} "
          f"AP={ap:.3f} ROC={roc:.3f} LogLoss={nll:.3f} Brier={bri:.3f} ECE={ece:.3f}")
    print(pd.DataFrame(cm, index=["true 0","true 1"], columns=["pred 0","pred 1"]))

    # --- Save plots ---
    outdir = (diag_root / m)
    figdir = outdir / "figs"
    outdir.mkdir(parents=True, exist_ok=True)
    save_all_eval_figures(figdir, y_test, y_hat, p_test, class_names=["neg","pos"])

    # --- Save predictions & errors ---
    errs = X_test.copy()
    errs["y_true"] = y_test.values
    errs["y_prob"] = p_test
    errs["y_pred"] = y_hat
    errs["error_type"] = np.where((errs["y_true"] == 1) & (errs["y_pred"] == 0), "FN",
                          np.where((errs["y_true"] == 0) & (errs["y_pred"] == 1), "FP", "OK"))
    try:
        errs["session"] = session_test.values
    except Exception:
        pass
    errs.to_csv(outdir / "predictions_with_errors.csv", index=False)
    errs[errs["error_type"].isin(["FP","FN"])].to_csv(outdir / "misclassified_only.csv", index=False)

    # --- Permutation importances on final calibrated model ---
    perm_auc = _permutation_importance_df(
        cal, use_X_perm, use_y_perm, feat_names,
        mode="roc_auc", n_repeats=30
    )
    perm_auc.to_csv(outdir / f"importance_perm_rocauc_on_{perm_where}.csv", index=False)

    # Optional: AP-based permutation importances (threshold-free)
    # perm_ap = _permutation_importance_df(
    #     cal, use_X_perm, use_y_perm, feat_names,
    #     mode="average_precision", n_repeats=30
    # )
    # perm_ap.to_csv(outdir / f"importance_perm_ap_on_{perm_where}.csv", index=False)

    # At your operating threshold (diagnostic only)
    perm_f1thr = _permutation_importance_df(
        cal, use_X_perm, use_y_perm, feat_names,
        mode="f1_at_threshold", threshold=thr, n_repeats=30
    )
    perm_f1thr.to_csv(outdir / f"importance_perm_f1_thr{thr:.3f}_on_{perm_where}.csv", index=False)

    # --- Model-native importances ---
    if 'best_models' in globals() and m in best_models:
        base = best_models[m]
        base_fitted = base.fit(X_train_full, y_train_full)

        coef_df = _linear_coef_df(base_fitted, feat_names)
        if coef_df is not None:
            coef_df.to_csv(outdir / "importance_coefficients.csv", index=False)

        tree_df = _tree_importance_df(base_fitted, feat_names)
        if tree_df is not None:
            tree_df.to_csv(outdir / "importance_impurity.csv", index=False)

    # --- Config snapshot (from Step 4 entry + new test metrics) ---
    tn, fp, fn, tp = cm.ravel()
    meta = {
        "model": m,
        "calibration": method,
        "threshold": thr,
        "val_metrics_from_step4": entry.get("val_metrics", {}),
        "test_metrics": {
            "precision": float(prec),
            "recall": float(rec),
            "f1": float(f1),
            "AP": float(ap),
            "ROC_AUC": float(roc),
            "LogLoss": float(nll),
            "Brier": float(bri),
            "ECE": float(ece),
            "confusion_matrix": [int(tn), int(fp), int(fn), int(tp)]
        }
    }
    (outdir / "config_metrics.json").write_text(json.dumps(meta, indent=2))

    # summary row
    summary_rows.append({
        "model": m, "calibration": method, "threshold": thr,
        "test_LogLoss": float(nll), "test_Brier": float(bri), "test_ECE": float(ece),
        "test_AP": float(ap), "test_ROC_AUC": float(roc),
        "test_precision": float(prec), "test_recall": float(rec), "test_F1": float(f1),
        "perm_top_feature_rocauc": (perm_auc.iloc[0]["feature"] if len(perm_auc) else None),
        "perm_top_feature_f1thr": (perm_f1thr.iloc[0]["feature"] if len(perm_f1thr) else None),
    })

# Roll-up table (probability-first sort)
if summary_rows:
    summary_df = (
        pd.DataFrame(summary_rows)
          .sort_values(["test_LogLoss", "test_Brier"], ascending=[True, True])
          .reset_index(drop=True)
    )
    display(summary_df)
    summary_df.to_csv(diag_root / "summary_across_models.csv", index=False)
    print(f"\n[INFO] Diagnostics written under: {diag_root}")
else:
    print("[WARN] No models diagnosed — check Step 4 outputs in", final_dir)


[INFO] Permutation importance will run on: val set

=== LogisticRegression | load final model | calibration=sigmoid | threshold=0.600 ===
[TEST] P=0.626 R=0.969 F1=0.761 AP=0.564 ROC=0.409 LogLoss=0.667 Brier=0.237 ECE=0.039
        pred 0  pred 1
true 0       0      37
true 1       2      62


/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")



=== RandomForest | load final model | calibration=sigmoid | threshold=0.050 ===
[TEST] P=0.634 R=1.000 F1=0.776 AP=0.639 ROC=0.413 LogLoss=0.676 Brier=0.241 ECE=0.138
        pred 0  pred 1
true 0       0      37
true 1       0      64


/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")



=== GaussianNB | load final model | calibration=sigmoid | threshold=0.050 ===
[TEST] P=0.634 R=1.000 F1=0.776 AP=0.563 ROC=0.378 LogLoss=0.666 Brier=0.236 ECE=0.047
        pred 0  pred 1
true 0       0      37
true 1       0      64


...

/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")



=== SVC_rbf | load final model | calibration=isotonic | threshold=0.510 ===
[TEST] P=0.634 R=1.000 F1=0.776 AP=0.616 ROC=0.433 LogLoss=0.665 Brier=0.236 ECE=0.038
        pred 0  pred 1
true 0       0      37
true 1       0      64


/opt/anaconda3/envs/RVL/lib/python3.12/site-packages/sklearn/metrics/_plot/roc_curve.py:189: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  self.ax_.legend(loc="lower right")


,model,calibration,threshold,test_LogLoss,test_Brier,test_ECE,test_AP,test_ROC_AUC,test_precision,test_recall,test_F1,perm_top_feature_rocauc,perm_top_feature_f1thr
0,KNN,sigmoid,0.05,0.659649,0.233337,0.025004,0.634029,0.495777,0.633663,1.00000,0.775758,follow_duration,windup_duration
1,SVC_rbf,isotonic,0.51,0.664887,0.235631,0.038381,0.616481,0.433066,0.633663,1.00000,0.775758,follow_duration,windup_duration
2,GaussianNB,sigmoid,0.05,0.666301,0.236254,0.046640,0.563417,0.377534,0.633663,1.00000,0.775758,windup_duration,windup_duration
3,LinearSVC,sigmoid,0.60,0.666648,0.236506,0.038762,0.563559,0.408361,0.626263,0.96875,0.760736,windup_duration,windup_duration
4,LogisticRegression,sigmoid,0.60,0.666746,0.236552,0.039028,0.563782,0.408784,0.626263,0.96875,0.760736,windup_duration,windup_duration
5,GradientBoosting,sigmoid,0.05,0.673067,0.239087,0.083959,0.592268,0.439189,0.633663,1.00000,0.775758,total_duration,windup_duration
6,RandomForest,sigmoid,0.05,0.676024,0.240954,0.137905,0.638845,0.412584,0.633663,1.00000,0.775758,follow_duration,windup_duration



[INFO] Diagnostics written under: /Users/kwill55/RVL/personalized-sports-optimization/experiments/summary_stats/2025-08-26_14-05-56_run-001_baseline/diagnostics_final


In [66]:
# ===== Step 6: Feature Pruning from Permutation Importance =====

#sel_ang_and_static
EXPERIMENT = "2025-08-26_14-05-56_run-001_all_angles_all_stats" # <-- adjust path

# --- Config ---
EXPERIMENT_DIR = Path("../experiments") / MODEL_TYPE / EXPERIMENT 
IMPORTANCE_DIR = EXPERIMENT_DIR / "diagnostics_final"
OUTPUT_LIST    = EXPERIMENT_DIR / "reduced_features.csv"
OUTPUT_MERGED  = EXPERIMENT_DIR / "feature_importance_merged.csv"
THRESHOLD      = 0.01  # keep if mean_importance >= THRESHOLD after clipping negatives to 0

# Recursively find permutation importance CSVs under each model dir
files = list(IMPORTANCE_DIR.glob("**/importance_perm_*_on_val.csv"))
if not files:
    raise FileNotFoundError(f"No permutation CSVs found under {IMPORTANCE_DIR} (**/importance_perm_*_on_val.csv)")

# ---- Load all to long-form ----
rows = []
for f in files:
    # model name is the immediate parent directory (e.g., LinearSVC)
    model = f.parent.name
    # metric is everything between 'importance_perm_' and '_on_val'
    m = re.search(r"importance_perm_(.+)_on_val\.csv$", f.name)
    metric = m.group(1) if m else "unknown"

    df = pd.read_csv(f)  # expects: feature, perm_importance_mean, perm_importance_std
    if {"feature","perm_importance_mean","perm_importance_std"} - set(df.columns):
        raise ValueError(f"Unexpected columns in {f}: {df.columns.tolist()}")

    tmp = df[["feature","perm_importance_mean","perm_importance_std"]].copy()
    tmp["metric"] = metric
    tmp["model"]  = model
    rows.append(tmp)

long_df = pd.concat(rows, ignore_index=True)
long_df["perm_importance_mean"] = long_df["perm_importance_mean"].clip(lower=0)

# ---- Aggregate across models for each (feature, metric) ----
agg_df = (
    long_df
    .groupby(["feature","metric"], as_index=False)
    .agg(mean_over_models=("perm_importance_mean","mean"),
         std_over_models=("perm_importance_mean","std"))
)

# ---- Pivot to wide: one row per feature, columns = metrics (averaged over models) ----
wide = agg_df.pivot_table(index="feature", columns="metric", values="mean_over_models", aggfunc="first").fillna(0.0)
# flatten column index and prefix with 'mean_'
wide.columns = [f"mean_{c}" for c in wide.columns]
wide = wide.reset_index()

# ---- Overall mean importance across all metrics ----
mean_cols = [c for c in wide.columns if c.startswith("mean_")]
wide["mean_importance"] = wide[mean_cols].mean(axis=1)

# ---- Filter by threshold and save ----
reduced = wide[wide["mean_importance"] >= THRESHOLD].sort_values("mean_importance", ascending=False)
reduced["feature"].to_csv(OUTPUT_LIST, index=False)
wide.sort_values("mean_importance", ascending=False).to_csv(OUTPUT_MERGED, index=False)

print(f"[INFO] Files loaded: {len(files)}")
print(f"[INFO] Features kept: {len(reduced)} / {len(wide)} (threshold={THRESHOLD})")
print(f"[INFO] Saved reduced list -> {OUTPUT_LIST}")
print(f"[INFO] Saved merged table -> {OUTPUT_MERGED}")


[INFO] Files loaded: 14
[INFO] Features kept: 0 / 3 (threshold=0.01)
[INFO] Saved reduced list -> ../experiments/summary_stats/2025-08-26_14-05-56_run-001_all_angles_all_stats/reduced_features.csv
[INFO] Saved merged table -> ../experiments/summary_stats/2025-08-26_14-05-56_run-001_all_angles_all_stats/feature_importance_merged.csv
